# Jacobians & Vector Calculus

**Companion lesson:** https://ml-viz.vercel.app/courses/calculus-for-ml/04-jacobians

A from-scratch, runnable derivation of Jacobians for the layers you use every day.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
np.random.seed(0)

## The Jacobian: definition and shape

The Jacobian of $\mathbf{f}: \mathbb{R}^n \to \mathbb{R}^m$ is the $m \times n$ matrix $J_{ij} = \partial f_i / \partial x_j$.
It linearly approximates $\mathbf{f}$ near any point: $\mathbf{f}(\mathbf{x}+\boldsymbol{\delta}) \approx \mathbf{f}(\mathbf{x}) + \mathbf{J}\boldsymbol{\delta}$.

We compute Jacobians three ways — by hand (pure stdlib), by finite differences, and by autograd (numpy-style) — and confirm they agree.

In [ ]:
import math

# --- Affine layer: f(x) = Wx + b ---
# Jacobian should equal W exactly
W = [[2.0, -1.0, 0.5],
     [0.0,  1.0, 3.0]]   # 2 outputs, 3 inputs
b = [1.0, -0.5]

def affine(x):
    return [sum(W[i][j] * x[j] for j in range(3)) + b[i] for i in range(2)]

# Finite-difference Jacobian (central differences, h=1e-5)
def fd_jacobian(f, x, h=1e-5):
    n = len(x)
    y0 = f(x)
    m = len(y0)
    J = [[0.0]*n for _ in range(m)]
    for j in range(n):
        xp = x[:]; xp[j] += h
        xm = x[:]; xm[j] -= h
        fp = f(xp); fm = f(xm)
        for i in range(m):
            J[i][j] = (fp[i] - fm[i]) / (2*h)
    return J

x0 = [1.0, 2.0, 3.0]
J_fd = fd_jacobian(affine, x0)

print('Finite-difference Jacobian of affine layer:')
for row in J_fd:
    print(' ', [round(v, 6) for v in row])
print('Weight matrix W:')
for row in W:
    print(' ', row)

# The Jacobian of an affine map is exactly its weight matrix W
assert all(abs(J_fd[i][j] - W[i][j]) < 1e-8 for i in range(2) for j in range(3)), \
    'Jacobian of affine layer must equal W'
print('VERIFY: J_affine == W:', True)

## Element-wise activations: diagonal Jacobian

For $\mathbf{f}(\mathbf{x}) = (\phi(x_1), \ldots, \phi(x_n))$ (same scalar $\phi$ applied independently), output $i$ only depends on input $i$, so $J_{ij} = \phi'(x_i) \cdot \mathbb{1}[i=j]$. The Jacobian is diagonal. Verify for ReLU and tanh.

In [ ]:
def relu(x): return [max(0.0, xi) for xi in x]
def tanh_vec(x): return [math.tanh(xi) for xi in x]

x0 = [-1.5, 0.5, 2.0, -0.3]
J_relu = fd_jacobian(relu, x0)
J_tanh = fd_jacobian(tanh_vec, x0)

# Off-diagonal entries must be ~0
for name, J in [('ReLU', J_relu), ('tanh', J_tanh)]:
    n = len(x0)
    off = sum(abs(J[i][j]) for i in range(n) for j in range(n) if i != j)
    diag = [J[i][i] for i in range(n)]
    print(f'{name}: diagonal = {[round(d,4) for d in diag]}, off-diagonal sum = {off:.2e}')

# ReLU diagonal: 1 where x>0, 0 elsewhere
relu_diag = [J_relu[i][i] for i in range(4)]
assert [round(d) for d in relu_diag] == [0, 1, 1, 0], \
    'ReLU Jacobian diagonal should be indicator of x>0'

# tanh diagonal: 1-tanh(xi)^2 (the sech^2 derivative)
tanh_diag = [J_tanh[i][i] for i in range(4)]
expected_tanh_diag = [1 - math.tanh(xi)**2 for xi in x0]
assert all(abs(tanh_diag[i] - expected_tanh_diag[i]) < 1e-6 for i in range(4)), \
    'tanh Jacobian diagonal should be 1 - tanh(x)^2'

print('VERIFY: element-wise activations have diagonal Jacobians.')

## Softmax Jacobian: the coupled case

Softmax is not independent per output — all outputs share the same normaliser. The Jacobian is dense:
$\mathbf{J} = \operatorname{diag}(\boldsymbol{\sigma}) - \boldsymbol{\sigma}\boldsymbol{\sigma}^\top$.

Verify numerically for the two-class case $\mathbf{x} = (1, 0)$ from the lesson.

In [ ]:
def softmax(x):
    e = [math.exp(xi - max(x)) for xi in x]   # numerically stable
    z = sum(e)
    return [ei / z for ei in e]

x0 = [1.0, 0.0]
s = softmax(x0)
print('softmax([1, 0]) =', [round(si, 4) for si in s])

J_sm_fd   = fd_jacobian(softmax, x0)
J_sm_anal = [[s[i]*(1 - s[j]) if i == j else -s[i]*s[j]
              for j in range(2)] for i in range(2)]

print('FD Jacobian:       ', [[round(v, 4) for v in row] for row in J_sm_fd])
print('Analytic Jacobian: ', [[round(v, 4) for v in row] for row in J_sm_anal])

assert all(abs(J_sm_fd[i][j] - J_sm_anal[i][j]) < 1e-7 for i in range(2) for j in range(2)), \
    'FD and analytic softmax Jacobians should agree'

# Rows must sum to 0 (probabilities sum to 1)
row_sums = [sum(J_sm_anal[i]) for i in range(2)]
assert all(abs(s) < 1e-12 for s in row_sums), 'Rows of softmax Jacobian must sum to 0'
print('VERIFY: rows sum to zero (probabilities are constrained to sum to 1).')

## Chain rule: Jacobian of a composed network

For $\mathbf{h} = \mathbf{g} \circ \mathbf{f}$, the Jacobian is the **product** $\mathbf{J}_{\mathbf{g}} \cdot \mathbf{J}_{\mathbf{f}}$ (computed at the appropriate points). Verify for a two-layer network: linear → ReLU → linear.

In [ ]:
# Two-layer network: R^3 -> R^2 -> R^2 -> R^1
# Layer 1: R^3 -> R^2 (affine + ReLU)
# Layer 2: R^2 -> R^1 (affine, no activation)

W1 = [[1.0, -0.5, 0.3], [0.2, 1.0, -1.0]]; b1 = [0.1, -0.2]
W2 = [[0.8, -0.6]]; b2 = [0.0]

def layer1(x): return [max(0.0, sum(W1[i][j]*x[j] for j in range(3)) + b1[i]) for i in range(2)]
def layer2(y): return [sum(W2[i][j]*y[j] for j in range(2)) + b2[i] for i in range(1)]
def network(x): return layer2(layer1(x))

x0 = [0.5, -1.0, 2.0]

# Full Jacobian by FD
J_full = fd_jacobian(network, x0)

# Via chain rule: J_L2 @ J_L1 (each computed at the appropriate intermediate point)
z1 = layer1(x0)
J1 = fd_jacobian(layer1, x0)     # 2x3
J2 = fd_jacobian(layer2, z1)     # 1x2

# matrix multiply J2 (1x2) @ J1 (2x3) -> (1x3)
J_chain = [[sum(J2[i][k]*J1[k][j] for k in range(2)) for j in range(3)] for i in range(1)]

print('J via FD on full network:', [[round(v, 6) for v in row] for row in J_full])
print('J via chain rule:        ', [[round(v, 6) for v in row] for row in J_chain])

assert all(abs(J_full[i][j] - J_chain[i][j]) < 1e-7
           for i in range(1) for j in range(3)), 'Chain rule must equal direct FD'
print('VERIFY: J_network = J_layer2 @ J_layer1 (vector chain rule).')

## VJP vs full Jacobian — cost comparison

Backpropagation computes $\mathbf{v}^\top \mathbf{J}$ (vector-Jacobian product) in a single backward pass, rather than computing all $mn$ entries of $\mathbf{J}$. For large $n, m$, this is the difference between feasibility and impossibility.

In [ ]:
import time

for n, m in [(100, 100), (500, 500), (1000, 1000)]:
    W_large = np.random.randn(m, n) * 0.1
    x_large = np.random.randn(n)
    v_large = np.random.randn(m)

    # VJP: v^T J = v^T W = W^T v  (one matrix-vector multiply)
    t0 = time.perf_counter()
    for _ in range(200):
        vjp = W_large.T @ v_large
    t_vjp = (time.perf_counter() - t0) / 200 * 1e6

    # Full Jacobian is just W (exact), but for a non-linear layer we'd need m forward passes
    # Simulate the cost: m forward passes through a vector of length n
    t0 = time.perf_counter()
    for _ in range(50):
        J_full_sim = np.zeros((m, n))
        for i in range(min(m, 20)):   # only 20 rows to keep this fast
            e_i = np.zeros(m); e_i[i] = 1.0
            J_full_sim[i] = W_large.T @ e_i  # proxy for one JVP
    t_full = (time.perf_counter() - t0) / 50 * 1e6 * (m / 20)

    print(f'n={n}, m={m}: VJP ≈ {t_vjp:.1f} µs | full Jacobian ≈ {t_full:.0f} µs (est.)')

print()
print('VJP cost is O(n), full Jacobian is O(mn) — the key reason backprop is efficient.')

## Condition number and gradient flow

The condition number $\kappa(\mathbf{J}) = \sigma_{\max}/\sigma_{\min}$ (ratio of singular values) measures how differently the layer treats different input directions.
A high condition number causes gradients to vanish in some directions and explode in others — the central cause of training instability in deep networks.

In [ ]:
# Xavier / Glorot initialization keeps condition number near 1 at init
# Compare with poorly-scaled initialization

def condition_number(W):
    sv = np.linalg.svd(W, compute_uv=False)
    return sv[0] / sv[-1] if sv[-1] > 1e-12 else float('inf')

np.random.seed(1)
n_layers = 6

kappas_xavier = []
kappas_poor = []
for _ in range(n_layers):
    d = 64
    W_xav  = np.random.randn(d, d) * np.sqrt(2.0 / (d + d))  # Xavier
    W_poor = np.random.randn(d, d) * 2.0                      # too large
    kappas_xavier.append(condition_number(W_xav))
    kappas_poor.append(condition_number(W_poor))

fig, ax = plt.subplots()
ax.plot(kappas_xavier, 'o-', color='#6366f1', label='Xavier init')
ax.plot(kappas_poor, 's--', color='#f59e0b', label='Large init (×2)')
ax.set_xlabel('layer index'); ax.set_ylabel('condition number κ(J)')
ax.set_title('Condition number per layer: Xavier vs large initialization')
ax.legend(); plt.show()

print('Xavier mean κ: {:.2f} | Large-init mean κ: {:.2f}'.format(
    sum(kappas_xavier)/len(kappas_xavier),
    sum(kappas_poor)/len(kappas_poor)
))

## Key takeaways

- **Jacobian shape**: $\mathbf{f}: \mathbb{R}^n \to \mathbb{R}^m$ → $\mathbf{J} \in \mathbb{R}^{m \times n}$ (outputs × inputs).
- **Affine layer**: $\mathbf{J} = \mathbf{W}$; **element-wise**: diagonal; **softmax**: $\operatorname{diag}(\boldsymbol{\sigma}) - \boldsymbol{\sigma}\boldsymbol{\sigma}^\top$.
- **Vector chain rule**: $\mathbf{J}_{\mathbf{g}\circ\mathbf{f}} = \mathbf{J}_{\mathbf{g}} \cdot \mathbf{J}_{\mathbf{f}}$ — backprop is this product computed efficiently.
- **VJP** $\mathbf{v}^\top \mathbf{J}$ costs $O(n)$; full Jacobian costs $O(mn)$ — the key to scalable backprop.
- **Condition number** $\kappa(\mathbf{J})$ governs gradient flow; Xavier initialization targets $\kappa \approx 1$.

## ✏️ Your turn

### Exercise 1 — Jacobian of the softmax (from scratch)

The softmax Jacobian is $\mathbf{J} = \operatorname{diag}(\boldsymbol{\sigma}) - \boldsymbol{\sigma}\boldsymbol{\sigma}^\top$.

Implement it and verify the key structural properties: rows sum to 0, symmetry, and exact match with a finite-difference estimate.

In [ ]:
import numpy as np

def softmax_jacobian(x):
    """Analytic Jacobian of softmax at x (1-D array).
    Returns J of shape (n, n) where J = diag(sigma) - sigma @ sigma.T."""
    # TODO(you): compute softmax probabilities sigma, then build the Jacobian
    ...

In [ ]:
x = np.array([1.0, 2.0, 0.5])
J = softmax_jacobian(x)

assert J.shape == (3, 3), "Jacobian must be (n, n)"
assert np.allclose(J.sum(axis=1), 0.0, atol=1e-9), \
    "rows of the softmax Jacobian must sum to 0 (probabilities sum to 1)"
assert np.allclose(J, J.T, atol=1e-12), \
    "softmax Jacobian is symmetric"

# Finite-difference check
def sm(x): e = np.exp(x - x.max()); return e / e.sum()
h = 1e-5
J_fd = np.zeros((3, 3))
for j in range(3):
    xp, xm = x.copy(), x.copy()
    xp[j] += h; xm[j] -= h
    J_fd[:, j] = (sm(xp) - sm(xm)) / (2*h)
assert np.allclose(J, J_fd, atol=1e-6), \
    "analytic Jacobian must match finite-difference estimate"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def softmax_jacobian(x):
    e = np.exp(x - x.max())
    sigma = e / e.sum()
    return np.diag(sigma) - np.outer(sigma, sigma)
```

</details>

### Exercise 2 — VJP of the affine layer

For $\mathbf{y} = \mathbf{W}\mathbf{x} + \mathbf{b}$ with $\mathbf{W} \in \mathbb{R}^{m \times n}$, the Jacobian is $\mathbf{W}$.
The VJP with upstream gradient $\mathbf{v} \in \mathbb{R}^m$ is $\mathbf{v}^\top \mathbf{J} = \mathbf{v}^\top \mathbf{W}$, which equals $\mathbf{W}^\top \mathbf{v}$ as a column vector.

Implement it and verify it matches the finite-difference gradient.

In [ ]:
def affine_vjp(W, x, b, v):
    """VJP of y = Wx + b with upstream gradient v.
    Returns grad_x = W^T v (same shape as x)."""
    # TODO(you): one line
    ...

In [ ]:
import numpy as np

np.random.seed(7)
m_dim, n_dim = 4, 6
W2 = np.random.randn(m_dim, n_dim)
b2 = np.random.randn(m_dim)
x2 = np.random.randn(n_dim)
v2 = np.random.randn(m_dim)

grad_x = affine_vjp(W2, x2, b2, v2)

assert grad_x.shape == x2.shape, \
    "VJP output must have same shape as x"
assert np.allclose(grad_x, W2.T @ v2, atol=1e-12), \
    "VJP must equal W^T @ v"

# Finite-difference check: grad_x[j] = d/dx_j (v^T (Wx+b))
loss = lambda x: float(v2 @ (W2 @ x + b2))
h = 1e-5
grad_fd = np.array([(loss(x2 + h*np.eye(n_dim)[j]) - loss(x2 - h*np.eye(n_dim)[j])) / (2*h)
                    for j in range(n_dim)])
assert np.allclose(grad_x, grad_fd, atol=1e-7), \
    "VJP must match finite-difference gradient of v^T(Wx+b) w.r.t. x"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def affine_vjp(W, x, b, v):
    return W.T @ v
```

</details>

### Exercise 3 — Condition number and singular values

The condition number of a layer's Jacobian is $\kappa = \sigma_{\max} / \sigma_{\min}$.
Xavier initialization targets $\kappa \approx 1$ by scaling weights by $\sqrt{2/(n_{in}+n_{out})}$.

Compute the condition number and verify that Xavier-initialized layers are better conditioned than large-initialized ones.

In [ ]:
import numpy as np

def condition_number(W):
    """Return the condition number sigma_max / sigma_min of matrix W.
    Use np.linalg.svd to get singular values."""
    # TODO(you): compute singular values, return max/min
    ...

In [ ]:
np.random.seed(42)
d = 64

W_xavier = np.random.randn(d, d) * np.sqrt(2.0 / (d + d))
W_large  = np.random.randn(d, d) * 2.0
W_identity = np.eye(d)

kappa_xav  = condition_number(W_xavier)
kappa_large = condition_number(W_large)
kappa_eye  = condition_number(W_identity)

assert abs(kappa_eye - 1.0) < 1e-9, \
    "identity matrix has condition number 1 (all singular values equal 1)"
assert kappa_xav < kappa_large, \
    "Xavier initialization gives smaller condition number than large-scale random init"
assert kappa_xav < 20, \
    "Xavier condition number should be modest (typically < 10–20 for d=64)"
print(f"κ(Xavier) = {kappa_xav:.2f} | κ(large) = {kappa_large:.2f} | κ(identity) = {kappa_eye:.2f}")
print("✅ Exercise 3 passed")

<details>
<summary>💡 Show solution</summary>

```python
def condition_number(W):
    sv = np.linalg.svd(W, compute_uv=False)
    return sv[0] / sv[-1]
```

</details>